[← 돌아가기](00.index.md)

# 07강. 파이썬 날아오르기

## index

- [첫째, 파이썬과 유니코드](#첫째-파이썬과-유니코드)
  - [(1) 유니코드 이해하기](#1-유니코드-이해하기)
  - [(2) 문자열 인코딩과 디코딩](#2-문자열-인코딩과-디코딩)
- [둘째, 클로저와 데코레이터](#둘째-클로저와-데코레이터)
  - [(1) 클로저의 개념](#1-클로저의-개념)
  - [(2) 데코레이터 작성 및 사용](#2-데코레이터-작성-및-사용)
  - [(3) 매개변수가 있는 데코레이터](#3-매개변수가-있는-데코레이터)
  - [(4) 클래스 데코레이터 및 고급 기법](#4-클래스-데코레이터-및-고급-기법)
- [셋째, 이터레이터와 제너레이터](#셋째-이터레이터와-제너레이터)
  - [(1) 이터레이터 프로토콜](#1-이터레이터-프로토콜)
  - [(2) 제너레이터 함수](#2-제너레이터-함수)
- [넷째, 파이썬 타입 어노테이션](#넷째-파이썬-타입-어노테이션)
  - [(1) 타입 힌트 기본](#1-타입-힌트-기본)
  - [(2) 고급 타입 어노테이션](#2-고급-타입-어노테이션)

### 첫째, 파이썬과 유니코드
#### (1) 유니코드 이해하기

- 유니코드(Unicode): 전 세계 문자를 표현하는 국제 표준 문자 집합
- 코드 포인트(Code Point): 각 문자를 나타내는 고유한 번호 (U+0041 = 'A')
- UTF-8, UTF-16, UTF-32: 유니코드를 인코딩하는 방식
- 파이썬 3: 모든 문자열을 기본적으로 유니코드로 처리

In [ ]:
text = "Hello 안녕 🌍"
print(f"텍스트: {text}")
print(f"길이: {len(text)}")

for char in text:
    print(f"{char}: U+{ord(char):04X}")

char_from_code = chr(0x4E2D)
print(f"\n코드 포인트 U+4E2D: {char_from_code}")

#### (2) 문자열 인코딩과 디코딩

- 인코딩(Encoding): 문자열(str) → 바이트(bytes)
- 디코딩(Decoding): 바이트(bytes) → 문자열(str)
- `.encode()`: 문자열을 바이트로 변환
- `.decode()`: 바이트를 문자열로 변환
- 인코딩 오류 처리: `errors='ignore'`, `errors='replace'` 등

In [ ]:
text = "안녕하세요"
utf8_bytes = text.encode('utf-8')
print(f"UTF-8: {utf8_bytes}")

utf16_bytes = text.encode('utf-16')
print(f"UTF-16: {utf16_bytes}")

decoded = utf8_bytes.decode('utf-8')
print(f"디코딩된 텍스트: {decoded}")

wrong_bytes = b'\\xc3\\xa9'
print(f"디코딩: {wrong_bytes.decode('utf-8')}")

### 둘째, 클로저와 데코레이터
#### (1) 클로저의 개념

- 클로저(Closure): 함수가 정의된 외부 함수의 변수에 접근할 수 있는 함수
- 자유 변수(Free Variable): 클로저가 참조하는 외부 함수의 변수
- 중첩 함수(Nested Function): 함수 내부에 정의된 함수
- `nonlocal`: 중첩 함수에서 외부 함수의 변수를 수정

In [ ]:
def outer(x):
    def inner(y):
        return x + y
    return inner

add_5 = outer(5)
print(f"add_5(3) = {add_5(3)}")

def counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

my_counter = counter()
print(f"Count: {my_counter()}")
print(f"Count: {my_counter()}")
print(f"Count: {my_counter()}")

#### (2) 데코레이터 작성 및 사용 - 기본 원리

**데코레이터의 기본 개념**

- 데코레이터(Decorator): 함수나 클래스의 기능을 수정하거나 확장하는 함수
- 클로저를 활용한 함수 래핑(Wrapping)
- `@decorator` 문법: 데코레이터 적용의 간편한 방법
- `functools.wraps`: 데코레이터 작성 시 원본 함수 정보 보존

**파이썬에서 함수는 객체**

- 함수를 변수에 할당 가능
- 함수를 다른 함수의 인자로 전달 가능  
- 함수가 다른 함수를 반환 가능

In [ ]:
import functools
import time

# 1. 기본 데코레이터
def simple_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[시작] {func.__name__} 함수 호출")
        result = func(*args, **kwargs)
        print(f"[종료] {func.__name__} 함수 반환")
        return result
    return wrapper

@simple_decorator
def greet(name):
    return f"안녕, {name}!"

print(greet("Alice"))
print(f"함수명: {greet.__name__}")

In [ ]:
# 2. 실행 시간 측정 데코레이터
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        elapsed = end - start
        print(f"{func.__name__} 실행 시간: {elapsed:.4f}초")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(0.1)
    return "완료"

result = slow_function()

In [ ]:
# 3. 디버그 데코레이터 - 여러 함수에 재사용
def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"호출: {func.__name__}{args} {kwargs}")
        result = func(*args, **kwargs)
        print(f"반환: {result}")
        return result
    return wrapper

@debug
def add(a, b):
    return a + b

@debug
def multiply(a, b):
    return a * b

add(2, 3)
multiply(4, 5)

#### (3) 매개변수가 있는 데코레이터

**3단계 래핑 구조**

데코레이터에 매개변수를 전달하려면 추가 함수 층이 필요합니다.

- 1단계: 매개변수를 받는 함수
- 2단계: 실제 데코레이터 함수
- 3단계: 원본 함수를 래핑하는 wrapper

In [ ]:
# 1. 반복 횟수를 지정하는 데코레이터
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for i in range(times):
                result = func(*args, **kwargs)
                results.append(result)
            return results
        return wrapper
    return decorator

@repeat(times=3)
def say_hello():
    return "Hello!"

print(say_hello())

In [ ]:
# 2. 로깅 레벨을 지정하는 데코레이터
def log_level(level):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            prefix = {
                'DEBUG': '[DEBUG]',
                'INFO': '[INFO]',
                'WARNING': '[WARNING]',
                'ERROR': '[ERROR]'
            }
            print(f"{prefix.get(level, level)} {func.__name__} 호출")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@log_level('INFO')
def process_data():
    return "데이터 처리 완료"

@log_level('ERROR')
def error_function():
    return "오류 발생"

process_data()
error_function()

#### (4) 클래스 데코레이터 및 고급 기법

**클래스 메서드에 데코레이터 적용**

- 클래스 메서드도 데코레이터 적용 가능
- 입력 검증, 권한 확인, 로깅 등에 활용

**데코레이터 체이닝**

- 여러 데코레이터를 한 번에 적용
- 아래에서 위로 순서대로 실행

In [ ]:
# 1. 클래스 메서드에 데코레이터 적용
def validate_positive(func):
    @functools.wraps(func)
    def wrapper(self, value):
        if value <= 0:
            raise ValueError("양수만 허용됩니다")
        return func(self, value)
    return wrapper

class BankAccount:
    def __init__(self, balance):
        self._balance = balance
    
    @validate_positive
    def deposit(self, amount):
        self._balance += amount
        return f"입금: {amount}, 잔액: {self._balance}"
    
    @validate_positive
    def withdraw(self, amount):
        self._balance -= amount
        return f"출금: {amount}, 잔액: {self._balance}"

account = BankAccount(1000)
print(account.deposit(500))
try:
    account.withdraw(-100)
except ValueError as e:
    print(f"오류: {e}")

In [ ]:
# 2. 데코레이터 체이닝
def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper() if isinstance(result, str) else result
    return wrapper

def add_exclamation(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"{result}!" if isinstance(result, str) else result
    return wrapper

@uppercase
@add_exclamation
def greet(name):
    return f"hello {name}"

print(greet("alice"))

@add_exclamation
@uppercase
def greet2(name):
    return f"hello {name}"

print(greet2("bob"))

In [ ]:
# 3. 캐싱 데코레이터 (메모이제이션)
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key in cache:
            print(f"캐시에서 반환: {key}")
            return cache[key]
        result = func(*args, **kwargs)
        cache[key] = result
        return result
    return wrapper

@memoize
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(5))
print(fibonacci(5))

In [ ]:
# 4. 권한 확인 데코레이터
def require_permission(permission):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user, *args, **kwargs):
            if permission not in user.get('permissions', []):
                raise PermissionError(f"권한 부족: {permission} 필요")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@require_permission('admin')
def delete_user(user, user_id):
    return f"사용자 {user_id} 삭제됨"

@require_permission('edit')
def edit_post(user, post_id):
    return f"게시물 {post_id} 수정됨"

admin_user = {'name': 'Admin', 'permissions': ['admin', 'edit']}
regular_user = {'name': 'User', 'permissions': ['edit']}

print(delete_user(admin_user, 123))
try:
    delete_user(regular_user, 123)
except PermissionError as e:
    print(f"오류: {e}")

### 셋째, 이터레이터와 제너레이터
#### (1) 이터레이터 프로토콜

- 이터레이터(Iterator): `__iter__()`, `__next__()` 메서드를 구현한 객체
- `__iter__()`: 이터레이터 객체 반환
- `__next__()`: 다음 값 반환, 더 이상 없으면 `StopIteration` 예외 발생
- `iter()`, `next()`: 내장 함수

In [ ]:
class CountUp:
    def __init__(self, max):
        self.max = max
        self.current = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current < self.max:
            self.current += 1
            return self.current
        else:
            raise StopIteration

counter = CountUp(3)
print("next() 사용:")
print(next(counter))
print(next(counter))
print(next(counter))

counter2 = CountUp(3)
print("\nfor 루프 사용:")
for num in counter2:
    print(num)

#### (2) 제너레이터 함수

- 제너레이터(Generator): `yield` 키워드를 사용하는 함수
- 이터레이터보다 간단한 문법
- 메모리 효율적: 필요할 때마다 값을 생성
- `yield`: 값을 반환하고 함수 상태를 유지
- 제너레이터 표현식(Generator Expression)

In [ ]:
def count_up(max):
    current = 0
    while current < max:
        current += 1
        yield current

for num in count_up(3):
    print(num)

def fibonacci(n):
    a, b = 0, 1
    count = 0
    while count < n:
        yield a
        a, b = b, a + b
        count += 1

print("\n피보나치 수열:")
for fib in fibonacci(5):
    print(fib, end=" ")
print()

squares = (x**2 for x in range(5))
print("\n제너레이터 표현식 (제곱):")
for square in squares:
    print(square)

### 넷째, 파이썬 타입 어노테이션
#### (1) 타입 힌트 기본

- 타입 힌트(Type Hint): 변수, 함수의 매개변수, 반환값의 타입을 명시
- 파이썬 3.5+: 타입 힌트 기본 제공
- 런타임에 강제되지 않음, IDE와 정적 분석 도구가 활용
- 기본 타입: `int`, `str`, `float`, `bool`, `list`, `dict`, `tuple`
- `Optional`, `Union`: 여러 타입 표현

In [ ]:
from typing import Optional, Union, List, Dict

name: str = "Alice"
age: int = 30
height: float = 165.5
is_student: bool = False

def greet(name: str, age: int) -> str:
    return f"Hello, {name}! You are {age} years old."

print(greet("Alice", 30))

def get_age(name: str) -> Optional[int]:
    ages = {"Alice": 30, "Bob": 25}
    return ages.get(name)

print(f"Bob's age: {get_age('Bob')}")
print(f"Charlie's age: {get_age('Charlie')}")

def process(value: Union[str, int]) -> str:
    if isinstance(value, int):
        return f"Number: {value}"
    return f"Text: {value}"

print(process(42))
print(process("hello"))

#### (2) 고급 타입 어노테이션

- `List`, `Dict`, `Tuple`, `Set`: 컨테이너 타입 상세 지정
- `Callable`: 함수 타입
- `Any`: 모든 타입 허용
- `TypeVar`: 제네릭 타입 변수
- `Protocol`: 구조적 타입 핑
- `@dataclass`: 데이터 클래스 데코레이터

In [ ]:
from typing import List, Dict, Tuple, Callable, TypeVar, Generic
from dataclasses import dataclass

def process_numbers(numbers: List[int]) -> int:
    return sum(numbers)

def create_mapping(keys: List[str], values: List[int]) -> Dict[str, int]:
    return dict(zip(keys, values))

print(process_numbers([1, 2, 3, 4]))
print(create_mapping(["a", "b", "c"], [1, 2, 3]))

def apply_operation(a: int, b: int, op: Callable[[int, int], int]) -> int:
    return op(a, b)

add = lambda x, y: x + y
print(f"5 + 3 = {apply_operation(5, 3, add)}")

@dataclass
class Person:
    name: str
    age: int
    email: str

person = Person("Alice", 30, "alice@example.com")
print(f"Person: {person}")

T = TypeVar('T')

def first_element(seq: List[T]) -> T:
    return seq[0]

print(f"First int: {first_element([1, 2, 3])}")
print(f"First str: {first_element(['a', 'b', 'c'])}")